# SilverAI - Baseline Model Training

This notebook trains baseline classification and regression models for the 15-minute COMEX silver futures target. It uses only the exact 17 engineered features, preserves chronology, calibrates the direction threshold from training data only, evaluates validation data only, and does not perform feature selection.

## 1. Objective

Establish reproducible baseline models for `target_direction_15m` and `future_return_15m`. These results are diagnostic baselines, not evidence of profitability or reliable predictability. Detailed permutation-importance and ablation work belongs in Notebook 08.

## 2. Imports and Configuration

In [2]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
HORIZON_MINUTES = 15
PURGE_ROWS = 3
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path('..').resolve()
INPUT_PATH = PROJECT_ROOT / 'data' / 'silver_modeling_data.csv'
MODEL_ROOT = PROJECT_ROOT / 'saved_models'
CLASSIFIER_PATH = MODEL_ROOT / 'classifier'
REGRESSOR_PATH = MODEL_ROOT / 'regressor'
METADATA_PATH = MODEL_ROOT / 'metadata'
for path in [CLASSIFIER_PATH, REGRESSOR_PATH, METADATA_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print(f'Input: {INPUT_PATH}')
print(f'Model output root: {MODEL_ROOT}')

Input: C:\Users\ravik\Desktop\SilverAI\data\silver_modeling_data.csv
Model output root: C:\Users\ravik\Desktop\SilverAI\saved_models


## 3. Load Modeling Dataset

In [3]:
df = pd.read_csv(INPUT_PATH)
input_row_count = len(df)
required_columns = {
    'datetime_et', 'future_return_15m', 'target_direction_15m',
    'return_1', 'return_3', 'return_6', 'volatility_5', 'volatility_20', 'volatility_50',
    'sma_distance_20', 'sma_distance_50', 'candle_return', 'candle_range_pct',
    'upper_wick_ratio', 'lower_wick_ratio', 'relative_volume', 'volume_change',
    'hour_sin', 'hour_cos', 'day_of_week',
}
missing_required = required_columns.difference(df.columns)
if missing_required:
    raise ValueError(f'Missing required columns: {sorted(missing_required)}')

df['_timestamp_for_sorting'] = pd.to_datetime(df['datetime_et'], errors='coerce', utc=True)
if df['_timestamp_for_sorting'].isna().any():
    raise ValueError('Unparseable datetime_et values found.')
was_chronological = df['_timestamp_for_sorting'].is_monotonic_increasing
df = df.sort_values('_timestamp_for_sorting', kind='mergesort').reset_index(drop=True)

print(f'Input rows: {input_row_count:,}')
print(f'Input was already chronological: {was_chronological}')
print(f'Sorted timestamp range: {df["datetime_et"].iloc[0]} to {df["datetime_et"].iloc[-1]}')

Input rows: 160,612
Input was already chronological: True
Sorted timestamp range: 2023-11-01 02:10:00-04:00 to 2026-11-03 01:45:00-05:00


## 4. Data Validation

Rows are removed only when a required model input or target is missing or non-finite. Missing feature values are not replaced with zero because zero has a market meaning and could distort the baseline.

In [4]:
feature_columns = [
    'return_1', 'return_3', 'return_6',
    'volatility_5', 'volatility_20', 'volatility_50',
    'sma_distance_20', 'sma_distance_50',
    'candle_return', 'candle_range_pct', 'upper_wick_ratio', 'lower_wick_ratio',
    'relative_volume', 'volume_change', 'hour_sin', 'hour_cos', 'day_of_week',
]
regression_target = 'future_return_15m'
saved_classification_target = 'target_direction_15m'
raw_input_columns = {'open', 'high', 'low', 'close', 'volume', 'datetime_et'}
future_derived_columns = {'future_return_15m', 'target_direction_15m'}

numeric_check_columns = feature_columns + [regression_target]
missing_feature_rows = int(df[feature_columns].isna().any(axis=1).sum())
missing_target_rows = int(df[[regression_target, saved_classification_target]].isna().any(axis=1).sum())
infinite_rows = int(np.isinf(df[numeric_check_columns].to_numpy(dtype=float)).any(axis=1).sum())
rows_to_remove = df[feature_columns].isna().any(axis=1) | df[[regression_target, saved_classification_target]].isna().any(axis=1) | np.isinf(df[numeric_check_columns].to_numpy(dtype=float)).any(axis=1)
clean_df = df.loc[~rows_to_remove].reset_index(drop=True).copy()

if set(feature_columns) & raw_input_columns:
    raise AssertionError('Raw OHLCV/timestamp columns entered the feature list.')
if set(feature_columns) & future_derived_columns:
    raise AssertionError('Future-derived columns entered the feature list.')

print(f'Rows with missing feature values: {missing_feature_rows:,}')
print(f'Rows with missing target values: {missing_target_rows:,}')
print(f'Rows with infinite required values: {infinite_rows:,}')
print(f'Rows removed for data-quality reasons: {int(rows_to_remove.sum()):,}')
print(f'Clean rows: {len(clean_df):,}')
print('Saved provisional class distribution:', clean_df[saved_classification_target].value_counts().sort_index().to_dict())

Rows with missing feature values: 39
Rows with missing target values: 0
Rows with infinite required values: 0
Rows removed for data-quality reasons: 39
Clean rows: 160,573
Saved provisional class distribution: {-1: 59319, 0: 38350, 1: 62904}


## 5. Define Features and Targets

Notebook 06 stored labels using a provisional threshold. This notebook does not blindly reuse those labels. Instead, it calibrates a neutral band from the training segment only using a documented quantile of absolute training returns, then applies the unchanged threshold to validation and test rows. The calibration is descriptive, not optimized against validation or test performance.

In [5]:
X = clean_df[feature_columns].copy()
y_reg = clean_df[regression_target].astype(float).copy()

# Classification labels are created after the chronological split below.
# The saved Notebook 06 labels are retained for audit only, not used as final labels here.
assert X.columns.tolist() == feature_columns
assert not set(X.columns) & raw_input_columns
assert not set(X.columns) & future_derived_columns
print(f'Number of model input features: {X.shape[1]}')
print('Feature list:', feature_columns)

Number of model input features: 17
Feature list: ['return_1', 'return_3', 'return_6', 'volatility_5', 'volatility_20', 'volatility_50', 'sma_distance_20', 'sma_distance_50', 'candle_return', 'candle_range_pct', 'upper_wick_ratio', 'lower_wick_ratio', 'relative_volume', 'volume_change', 'hour_sin', 'hour_cos', 'day_of_week']


## 6. Chronological Data Splitting

The nominal split is 70% training, 15% validation, and 15% test. Three rows are purged from the end of training and validation so that target intervals do not cross the next partition boundary. The test partition is not used for fitting, threshold calibration, model comparison, or predictions.

In [6]:
n_rows = len(clean_df)
train_boundary = int(n_rows * TRAIN_FRACTION)
validation_boundary = int(n_rows * (TRAIN_FRACTION + VALIDATION_FRACTION))

train_raw = clean_df.iloc[:train_boundary].copy()
validation_raw = clean_df.iloc[train_boundary:validation_boundary].copy()
test = clean_df.iloc[validation_boundary:].copy()

if len(train_raw) <= PURGE_ROWS or len(validation_raw) <= PURGE_ROWS:
    raise ValueError('Partitions are too small for the required purge.')
train = train_raw.iloc[:-PURGE_ROWS].copy()
validation = validation_raw.iloc[:-PURGE_ROWS].copy()

def partition_summary(name, partition):
    return {
        'partition': name,
        'rows': len(partition),
        'start': partition['datetime_et'].iloc[0],
        'end': partition['datetime_et'].iloc[-1],
    }

partition_table = pd.DataFrame([
    partition_summary('train', train),
    partition_summary('validation', validation),
    partition_summary('test', test),
])
display(partition_table)
print(f'Purged rows from train boundary: {len(train_raw) - len(train)}')
print(f'Purged rows from validation boundary: {len(validation_raw) - len(validation)}')

,partition,rows,start,end
0,train,112398,2023-11-01 07:00:00-04:00,2025-11-28 12:05:00-05:00
1,validation,24083,2025-11-28 12:25:00-05:00,2026-05-06 08:15:00-04:00
2,test,24086,2026-05-06 08:35:00-04:00,2026-11-03 01:45:00-05:00


Purged rows from train boundary: 3
Purged rows from validation boundary: 3


In [7]:
TRAIN_NEUTRAL_QUANTILE = 0.40
training_returns = train[regression_target].astype(float)
calibrated_threshold_pct = float(training_returns.abs().quantile(TRAIN_NEUTRAL_QUANTILE))

def apply_direction_threshold(returns, threshold_pct):
    labels = pd.Series(np.nan, index=returns.index, dtype='float64')
    labels.loc[returns > threshold_pct] = 1
    labels.loc[returns < -threshold_pct] = -1
    labels.loc[returns.notna() & labels.isna()] = 0
    return labels.astype('int64')

train['classification_target'] = apply_direction_threshold(train[regression_target], calibrated_threshold_pct)
validation['classification_target'] = apply_direction_threshold(validation[regression_target], calibrated_threshold_pct)
test['classification_target'] = apply_direction_threshold(test[regression_target], calibrated_threshold_pct)

def class_distribution(partition):
    counts = partition['classification_target'].value_counts().reindex([-1, 0, 1], fill_value=0)
    percentages = (counts / len(partition) * 100).round(2)
    return pd.DataFrame({'count': counts, 'percentage': percentages})

print(f'Train-only calibration method: {TRAIN_NEUTRAL_QUANTILE:.0%} quantile of absolute training returns')
print(f'Calibrated threshold: +/-{calibrated_threshold_pct:.6f}%')
display(pd.concat({name: class_distribution(partition) for name, partition in [('train', train), ('validation', validation), ('test', test)]}, axis=1))

Train-only calibration method: 40% quantile of absolute training returns
Calibrated threshold: +/-0.077640%


train            validation             test           
                       count percentage      count percentage count percentage
classification_target                                                         
-1                     32423      28.85       9054      37.59  9060      37.62
 0                     44960      40.00       4751      19.73  6504      27.00
 1                     35015      31.15      10278      42.68  8522      35.38

## 7. Classification Model Training

All classifiers are fitted on training rows only. Logistic Regression uses a training-fitted imputer and scaler inside a pipeline; tree models use their native unscaled numeric inputs. Validation predictions are generated only after fitting.

In [8]:
X_train_cls = train[feature_columns]
X_validation_cls = validation[feature_columns]
y_train_cls = train['classification_target']
y_validation_cls = validation['classification_target']

classification_models = {
    'logistic_regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=SEED)),
    ]),
    'random_forest_classifier': RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=-1),
    'hist_gradient_boosting_classifier': HistGradientBoostingClassifier(max_iter=100, random_state=SEED),
}

classification_predictions = {}
classification_validation_metrics = []
classification_reports = {}
for model_name, model in classification_models.items():
    model.fit(X_train_cls, y_train_cls)
    predictions = model.predict(X_validation_cls)
    classification_predictions[model_name] = predictions
    report = classification_report(y_validation_cls, predictions, labels=[-1, 0, 1], output_dict=True, zero_division=0)
    classification_reports[model_name] = report
    classification_validation_metrics.append({
        'model': model_name,
        'accuracy': accuracy_score(y_validation_cls, predictions),
        'balanced_accuracy': balanced_accuracy_score(y_validation_cls, predictions),
        'macro_f1': f1_score(y_validation_cls, predictions, average='macro'),
        'down_precision': report['-1']['precision'],
        'down_recall': report['-1']['recall'],
        'down_f1': report['-1']['f1-score'],
        'neutral_precision': report['0']['precision'],
        'neutral_recall': report['0']['recall'],
        'neutral_f1': report['0']['f1-score'],
        'up_precision': report['1']['precision'],
        'up_recall': report['1']['recall'],
        'up_f1': report['1']['f1-score'],
    })

classification_validation_table = pd.DataFrame(classification_validation_metrics).sort_values('macro_f1', ascending=False)
display(classification_validation_table.round(4))

,model,accuracy,balanced_accuracy,macro_f1,down_precision,down_recall,down_f1,neutral_precision,neutral_recall,neutral_f1,up_precision,up_recall,up_f1
2,hist_gradient_boosting_classifier,0.4229,0.3870,0.3889,0.4013,0.3383,0.3671,0.3830,0.2414,0.2962,0.4441,0.5813,0.5035
1,random_forest_classifier,0.4157,0.3830,0.3870,0.3928,0.4036,0.3981,0.3651,0.2360,0.2867,0.4471,0.5094,0.4762
0,logistic_regression,0.4291,0.3955,0.3640,0.4061,0.1314,0.1986,0.3611,0.3075,0.3322,0.4492,0.7476,0.5612


## 8. Regression Model Training

In [9]:
X_train_reg = train[feature_columns]
X_validation_reg = validation[feature_columns]
y_train_reg = train[regression_target].astype(float)
y_validation_reg = validation[regression_target].astype(float)

regression_models = {
    'linear_regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', LinearRegression()),
    ]),
    'random_forest_regressor': RandomForestRegressor(n_estimators=150, random_state=SEED, n_jobs=-1),
    'hist_gradient_boosting_regressor': HistGradientBoostingRegressor(max_iter=100, random_state=SEED),
}

regression_predictions = {}
regression_validation_metrics = []
for model_name, model in regression_models.items():
    model.fit(X_train_reg, y_train_reg)
    predictions = model.predict(X_validation_reg)
    regression_predictions[model_name] = predictions
    regression_validation_metrics.append({
        'model': model_name,
        'mae': mean_absolute_error(y_validation_reg, predictions),
        'rmse': mean_squared_error(y_validation_reg, predictions) ** 0.5,
        'r2': r2_score(y_validation_reg, predictions),
    })

regression_validation_table = pd.DataFrame(regression_validation_metrics).sort_values('rmse')
display(regression_validation_table.round(6))

,model,mae,rmse,r2
0,linear_regression,0.357133,0.576681,-0.000234
2,hist_gradient_boosting_regressor,0.365830,0.589254,-0.044325
1,random_forest_regressor,0.371376,0.609040,-0.115635


## 9. Validation Performance Comparison

The baselines use no learned features: classification predicts the training majority class, and regression predicts a zero return. They are calculated without using test observations.

In [10]:
majority_class = int(y_train_cls.mode().iloc[0])
majority_predictions = np.full(len(y_validation_cls), majority_class)
classification_baseline = pd.DataFrame([{
    'model': 'majority_class_baseline',
    'accuracy': accuracy_score(y_validation_cls, majority_predictions),
    'balanced_accuracy': balanced_accuracy_score(y_validation_cls, majority_predictions),
    'macro_f1': f1_score(y_validation_cls, majority_predictions, average='macro'),
}])
classification_comparison = pd.concat([classification_baseline, classification_validation_table[['model', 'accuracy', 'balanced_accuracy', 'macro_f1']]], ignore_index=True)

zero_predictions = np.zeros(len(y_validation_reg))
regression_baseline = pd.DataFrame([{
    'model': 'zero_return_baseline',
    'mae': mean_absolute_error(y_validation_reg, zero_predictions),
    'rmse': mean_squared_error(y_validation_reg, zero_predictions) ** 0.5,
    'r2': r2_score(y_validation_reg, zero_predictions),
}])
regression_comparison = pd.concat([regression_baseline, regression_validation_table], ignore_index=True)

print('Classification validation comparison')
display(classification_comparison.round(4))
print('Regression validation comparison')
display(regression_comparison.round(6))

Classification validation comparison


,model,accuracy,balanced_accuracy,macro_f1
0,majority_class_baseline,0.1973,0.3333,0.1098
1,hist_gradient_boosting_classifier,0.4229,0.3870,0.3889
2,random_forest_classifier,0.4157,0.3830,0.3870
3,logistic_regression,0.4291,0.3955,0.3640


Regression validation comparison


,model,mae,rmse,r2
0,zero_return_baseline,0.357189,0.576632,-0.000064
1,linear_regression,0.357133,0.576681,-0.000234
2,hist_gradient_boosting_regressor,0.365830,0.589254,-0.044325
3,random_forest_regressor,0.371376,0.609040,-0.115635


## 10. Preliminary Feature Importance

The following rankings are descriptive only. Tree impurity importance can be biased, and correlated features can divide or transfer importance. Logistic coefficients are model-specific and are not causal evidence. Notebook 08 will handle out-of-sample permutation importance and ablation more carefully.

In [11]:
def tree_importance(model, names):
    return pd.Series(model.feature_importances_, index=names).sort_values(ascending=False).rename('importance')

preliminary_importance = {}
for model_name in ['random_forest_classifier', 'hist_gradient_boosting_classifier', 'random_forest_regressor', 'hist_gradient_boosting_regressor']:
    model = classification_models.get(model_name, regression_models.get(model_name))
    if hasattr(model, 'feature_importances_'):
        preliminary_importance[model_name] = tree_importance(model, feature_columns)

logistic_model = classification_models['logistic_regression'].named_steps['model']
preliminary_importance['logistic_regression_abs_coefficient'] = pd.Series(np.abs(logistic_model.coef_).mean(axis=0), index=feature_columns).sort_values(ascending=False).rename('mean_abs_coefficient')

for model_name, importance in preliminary_importance.items():
    print(model_name)
    display(importance.head(10).to_frame().round(6))

random_forest_classifier


,importance
volatility_20,0.073562
candle_range_pct,0.072972
volatility_5,0.069928
volatility_50,0.068645
sma_distance_50,0.066288
sma_distance_20,0.065105
relative_volume,0.063592
return_6,0.063524
volume_change,0.063167
return_3,0.063054


random_forest_regressor


,importance
volatility_50,0.075976
volatility_20,0.073637
volatility_5,0.072316
sma_distance_50,0.071699
return_3,0.069478
candle_range_pct,0.068825
return_6,0.068432
sma_distance_20,0.067508
relative_volume,0.064760
volume_change,0.058108


logistic_regression_abs_coefficient


,mean_abs_coefficient
candle_range_pct,0.109684
volatility_20,0.105714
hour_sin,0.045301
volatility_5,0.040354
relative_volume,0.038183
sma_distance_20,0.034382
volume_change,0.026379
upper_wick_ratio,0.017367
lower_wick_ratio,0.016863
volatility_50,0.016076


## 11. Save Models and Metadata

Only fitted models, configuration, feature names, target definitions, and train/validation metadata are saved. Test rows and test predictions are not saved as training artifacts.

In [12]:
saved_model_paths = {}
for model_name, model in classification_models.items():
    path = CLASSIFIER_PATH / f'{model_name}.joblib'
    joblib.dump(model, path)
    saved_model_paths[f'classifier:{model_name}'] = str(path)

for model_name, model in regression_models.items():
    path = REGRESSOR_PATH / f'{model_name}.joblib'
    joblib.dump(model, path)
    saved_model_paths[f'regressor:{model_name}'] = str(path)

metadata = {
    'feature_names': feature_columns,
    'feature_count': len(feature_columns),
    'raw_ohlcv_excluded': True,
    'prediction_horizon_minutes': HORIZON_MINUTES,
    'prediction_horizon_bars': 3,
    'classification_target': 'classification_target',
    'source_classification_target': saved_classification_target,
    'regression_target': regression_target,
    'classification_labels': {'-1': 'DOWN', '0': 'NEUTRAL', '1': 'UP'},
    'threshold_value_pct': calibrated_threshold_pct,
    'threshold_calibration_method': '40th percentile of absolute returns in purged training partition only',
    'threshold_is_provisional': True,
    'split_fractions_nominal': {'train': TRAIN_FRACTION, 'validation': VALIDATION_FRACTION, 'test': 0.15},
    'purge_rows_at_boundaries': PURGE_ROWS,
    'seed': SEED,
    'training_start': train['datetime_et'].iloc[0],
    'training_end': train['datetime_et'].iloc[-1],
    'validation_start': validation['datetime_et'].iloc[0],
    'validation_end': validation['datetime_et'].iloc[-1],
    'test_range_recorded_without_evaluation': {'start': test['datetime_et'].iloc[0], 'end': test['datetime_et'].iloc[-1]},
    'model_parameters': {
        'classification': {name: model.get_params() for name, model in classification_models.items()},
        'regression': {name: model.get_params() for name, model in regression_models.items()},
    },
    'saved_model_paths': saved_model_paths,
}
metadata_file = METADATA_PATH / 'training_metadata.json'
metadata_file.write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')
saved_model_paths['metadata'] = str(metadata_file)

# Verify every saved model can be loaded.
for key, path_string in saved_model_paths.items():
    if key != 'metadata':
        loaded = joblib.load(path_string)
        if not hasattr(loaded, 'predict'):
            raise AssertionError(f'Loaded object is not a model: {path_string}')
json.loads(metadata_file.read_text(encoding='utf-8'))

print('Saved and reloaded model artifacts:')
for path in saved_model_paths.values():
    print(path)

Saved and reloaded model artifacts:
C:\Users\ravik\Desktop\SilverAI\saved_models\classifier\logistic_regression.joblib
C:\Users\ravik\Desktop\SilverAI\saved_models\classifier\random_forest_classifier.joblib
C:\Users\ravik\Desktop\SilverAI\saved_models\classifier\hist_gradient_boosting_classifier.joblib
C:\Users\ravik\Desktop\SilverAI\saved_models\regressor\linear_regression.joblib
C:\Users\ravik\Desktop\SilverAI\saved_models\regressor\random_forest_regressor.joblib
C:\Users\ravik\Desktop\SilverAI\saved_models\regressor\hist_gradient_boosting_regressor.joblib
C:\Users\ravik\Desktop\SilverAI\saved_models\metadata\training_metadata.json


## 12. Final Summary

In [13]:
final_summary = pd.Series({
    'Dataset rows after data validation': len(clean_df),
    'Input feature count': len(feature_columns),
    'Training rows after purge': len(train),
    'Validation rows after purge': len(validation),
    'Test rows (not evaluated)': len(test),
    'Classification models trained': ', '.join(classification_models.keys()),
    'Regression models trained': ', '.join(regression_models.keys()),
    'Training-only threshold': f'+/-{calibrated_threshold_pct:.6f}%',
    'Threshold calibration': metadata['threshold_calibration_method'],
    'Validation best classification by macro F1': classification_validation_table.iloc[0]['model'],
    'Validation best regression by RMSE': regression_validation_table.iloc[0]['model'],
    'Metadata path': str(metadata_file),
})
display(final_summary.to_frame('value'))
print('Test observations were not scored and no test predictions were saved.')
print('These baselines do not establish profitability or reliable predictive power.')

,value
Dataset rows after data validation,160573
Input feature count,17
Training rows after purge,112398
Validation rows after purge,24083
Test rows (not evaluated),24086
Classification models trained,"logistic_regression, random_forest_classifier,..."
Regression models trained,"linear_regression, random_forest_regressor, hi..."
Training-only threshold,+/-0.077640%
Threshold calibration,40th percentile of absolute returns in purged ...
Validation best classification by macro F1,hist_gradient_boosting_classifier


Test observations were not scored and no test predictions were saved.
These baselines do not establish profitability or reliable predictive power.
